# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset on knowledge adoption predictors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is described by a Croissant schema and contains several record sets and fields describing survey and regression results for rangeland management practices in Northern Kenya.

### Dataset Source
The dataset source is provided as a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records for the dataset with `mlcroissant`. This fetches the package structure and descriptive metadata of the FAIR^2 dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant metadata and dataset object
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print('Dataset title: ' + meta.name)
print('Dataset description: ' + meta.description)
print('License: ' + meta.license)
print('Authors:')
for author in getattr(meta, 'author', []):
    print('  -', author['@id'])
print('---')
print('Available distribution ids:')
for dist in getattr(meta, 'distribution', []):
    print('  -', dist['@id'])

## 2. Data Overview

Now list the available record sets, with their `@id` and their available fields and columns (all by their `@id`). This will allow us to reference them in later steps.

In [ ]:
# Get all record sets by @id from the Croissant schema
record_sets_info = dataset.metadata.to_json().get('recordSet', [])

if not record_sets_info:
    print('No record sets defined explicitly in the top-level Croissant metadata. Trying dataset.record_sets...')
    record_sets = dataset.record_sets
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    record_set_ids = [rs['@id'] for rs in record_sets_info]

if not record_set_ids:
    # Fallback: try the record_sets attribute if to_json().get('recordSet', []) is empty
    record_set_objs = getattr(dataset, 'record_sets', [])
    record_set_ids = [rs['@id'] for rs in record_set_objs]
    record_sets_info = record_set_objs

if not record_set_ids:
    print('No record sets found in either metadata or dataset. The dataset may only have one default set (e.g., from a single table).')
else:
    print(f"Found {len(record_set_ids)} record set(s):\n")
    for rs in dataset.record_sets:
        print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields:")
            for field in fields:
                print(f"    - field @id: {field['@id']} (name: {field.get('name', '(no name)')})")
        if 'column' in rs:
            columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            print("  Columns:")
            for column in columns:
                print(f"    - column @id: {column['@id']} (name: {column.get('name', '(no name)')})")
        print()
        # Show snippet records if available
        try:
            sample_records = list(dataset.records(record_set=rs['@id']))
            print(f"  Sample record keys: {list(sample_records[0].keys()) if sample_records else 'No records.'}")
        except Exception as e:
            print(f"  Could not read records from: {rs['@id']} (error: {e})")

## 3. Data Extraction

Load data from each available record set into pandas DataFrames using the `mlcroissant.Dataset.records(record_set=...)` method. All entities are referenced by their `@id` fields.

In [ ]:
# Get the list of record set @ids
if not record_set_ids:
    # Fallback: try using 'default' as the record set
    record_set_ids = [None]

dataframes = {}
for rs_id in record_set_ids:
    try:
        # Use None for default record set if not present
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id if rs_id else 'default'] = df
        print(f"Loaded {len(df)} records for record set @id: {rs_id}")
        if len(df.columns):
            print('  Columns (shown by field @id):')
            print('   ', df.columns.tolist())
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Display first few rows of the first DataFrame
first_rs_id = record_set_ids[0] if record_set_ids else 'default'
if first_rs_id not in dataframes:
    first_rs_id = 'default'
display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

In this section, select a numeric field by its `@id` and perform basic filtering, normalization, and grouping. Replace the variable names with the correct `@id`s and with actual column names from your data set as needed.

In [ ]:
# Replace these variables below with the actual @id values from your overview cell above
# Example: numeric_field = 'http://mlcommons.org/croissant/fields/log_likelihood'
# In the absence of published field @ids, let's print all column @ids and choose one:

df = dataframes[first_rs_id]
print('Available fields (column @id):')
for col in df.columns:
    print('  -', col)
# Choose a numeric field (update this variable as needed)
numeric_field = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else None

if numeric_field:
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean()  # or any chosen threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean):")
    print(filtered_df.head())

    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Select a grouping field: pick the first object type field (if any)
    possible_group_fields = [c for c in df.columns if df[c].dtype == 'object']
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print('No numeric field found in data. Please update the field selection logic if your data contains numeric fields with string column names.')

## 5. Visualization

Visualize the distribution of the selected numeric field and any interesting relationships using matplotlib or seaborn. Adjust the field selection as needed based on your dataset's schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('No suitable numeric field found; cannot plot distribution.')

## 6. Conclusion

Using the `mlcroissant` library, we loaded and explored the FAIR^2 dataset. We examined record set and field `@id`s, loaded the data into DataFrames, filtered records, normalized and grouped numeric fields, and visualized key variables. This approach supports reproducible, standards-based analysis across complex tabular datasets described by the Croissant metadata schema.

**Next Steps:** For domain-specific insights, consult field and column `@id`s and refer to the Croissant schema for exact variable definitions, documentation, and downstream machine learning pipelines.